# AlexNet 垃圾分类：Colab 一键运行

在 Colab GPU 上运行全部实验：两个 HOG 传统基线 + 四个 AlexNet 训练配置，最后把 `results/` 打包下载。

**先切换到 GPU：** 菜单 *代码执行程序 → 更改运行时类型 → T4 GPU*，然后 *全部运行*。全部跑完在 T4 上约 30–40 分钟。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 克隆仓库并安装

In [ ]:
%cd /content
!rm -rf alexnet-waste-classification
!git clone -q https://github.com/weirdquantum/alexnet-waste-classification.git
%cd /content/alexnet-waste-classification
!pip install -q -e ".[dev]"
!python -m pytest -q

## 2. 下载 TrashNet 并划分 train / val / test（70 / 15 / 15，按类别分层）

In [ ]:
!mkdir -p data/download
!wget -q -O data/download/dataset-resized.zip https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip
!python scripts/prepare_trashnet.py

## 3. 传统基线：HOG + PCA + 决策树 / RBF-SVM（CPU，约 3 分钟）

In [ ]:
!python scripts/train_baseline.py

## 4. AlexNet 四种训练配置

| preset | 说明 |
|---|---|
| `course` | 原课程配方：192 px、像素 /255、无数据增强、Adam 1e-3 |
| `scratch` | AlexNet 从零训练 + 改进配方（ImageNet 归一化、数据增强、AdamW、label smoothing、warmup + cosine） |
| `scratch_bn` | 同上，但卷积后加 BatchNorm |
| `finetune` | ImageNet 预训练 AlexNet 微调 |

每个配置都按验证集 macro-F1 选最佳 epoch，测试集只在最后评估一次。

In [ ]:
for preset in ["course", "scratch", "scratch_bn", "finetune"]:
    print(f"===== {preset} =====", flush=True)
    !mkdir -p results/{preset}
    !python -u scripts/train.py --preset {preset} --num-workers 2 2>&1 | tee results/{preset}/train.log | grep -E "device=|best epoch|early stopping|Error|Traceback|epoch +[0-9]*[05] " 

## 5. 汇总结果

In [ ]:
!python scripts/summarize.py
import os
from IPython.display import Image, display
for name in ["hog_tree", "hog_svm", "course", "scratch", "scratch_bn", "finetune"]:
    print(name)
    for fig, width in [("confusion_matrix.png", 420), ("curves.png", 720)]:
        if os.path.exists(f"results/{name}/{fig}"):
            display(Image(f"results/{name}/{fig}", width=width))

## 6. 下载结果（不含模型权重）

下载的 `results.zip` 解压到项目根目录（覆盖 `results/`）即可。

In [ ]:
!zip -q -r results.zip results -x "*.pt" "*.joblib"
from google.colab import files
files.download("results.zip")